
# 05 — Prompt-based Judge + Simple Aggregator


In [ ]:

# Install once if needed:
# %pip install -q -U transformers accelerate torch pandas tqdm

import os
import re
import json
import ast
import pandas as pd
import numpy as np
import torch

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM


## 1. Paths and settings

In [ ]:

INPUT_CSV = r"../data/Prompt_Phi3_Claims_with_Evidence.csv"

CLAIM_OUTPUT_CSV = r"../data/Phi3_Claims_with_Prompt_Judgments.csv"
ANSWER_OUTPUT_CSV = r"../results/Phi3_Answer_Hallucination_Prompt_Judge.csv"

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

TOP_K_EVIDENCE = 3

print("Input:", INPUT_CSV)
print("Claim-level output:", CLAIM_OUTPUT_CSV)
print("Answer-level output:", ANSWER_OUTPUT_CSV)
print("Judge model:", MODEL_NAME)


## 2. Load retrieved claims

In [ ]:

df = pd.read_csv(INPUT_CSV)

print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())

required = {
    "Question_ID",
    "Question",
    "Phi3_Answer",
    "Claim_ID",
    "Atomic_Claim",
}

missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

evidence_cols = [
    f"Evidence_{i}"
    for i in range(1, TOP_K_EVIDENCE + 1)
    if f"Evidence_{i}" in df.columns
]

if not evidence_cols:
    raise ValueError("No Evidence_1 / Evidence_2 / ... columns found.")

print("Evidence columns used:", evidence_cols)

df.head()



## 3. Load Qwen Judge


In [ ]:

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("Model loaded.")
print("Device:", next(model.parameters()).device)


## 4. Prompt design

In [ ]:

SYSTEM_PROMPT = """
You are an evidence-based factuality judge.

Your task is to determine whether a factual claim is supported by the
provided evidence.

You must use ONLY the provided evidence.
Do NOT use your own background knowledge.

Labels:

SUPPORTED:
The evidence directly supports the important factual content of the claim.
Paraphrases and equivalent wording count as support.

UNSUPPORTED:
The evidence does not establish the claim, contradicts the claim, is
irrelevant, or is insufficient to verify the important factual content.

Important rules:
1. Judge the whole atomic claim.
2. Do not assume missing facts.
3. High topical similarity does NOT automatically mean support.
4. If the evidence discusses the same person/topic but does not establish
   the claimed fact, return UNSUPPORTED.
5. If no usable evidence is provided, return UNSUPPORTED.
6. Return valid JSON only.

Output exactly:
{
  "label": "SUPPORTED" or "UNSUPPORTED",
  "reason": "brief evidence-based explanation"
}
"""


def build_judge_messages(claim, evidence_text):
    user_prompt = f"""
Atomic Claim:
{claim}

Retrieved Evidence:
{evidence_text}

Determine whether the claim is SUPPORTED or UNSUPPORTED by the evidence.
"""

    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT.strip()
        },
        {
            "role": "user",
            "content": user_prompt.strip()
        }
    ]



## 5. Combine Top-k evidence


In [ ]:

def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()

    if not x:
        return ""

    if x.lower().startswith("error:"):
        return ""

    return x


def combine_evidence(row, evidence_cols=evidence_cols):
    pieces = []

    for i, col in enumerate(evidence_cols, start=1):
        text = clean_text(row.get(col, ""))

        if text:
            page_col = f"Evidence_{i}_Page"
            page = clean_text(row.get(page_col, ""))

            if page:
                pieces.append(
                    f"[Evidence {i} | Wikipedia page: {page}]\n{text}"
                )
            else:
                pieces.append(
                    f"[Evidence {i}]\n{text}"
                )

    if not pieces:
        return "[NO USABLE RETRIEVED EVIDENCE]"

    return "\n\n".join(pieces)


## 6. Robust JSON parser

In [ ]:

def parse_judgment(text):
    if not isinstance(text, str):
        return {
            "label": "UNSUPPORTED",
            "reason": "Judge returned no parseable output."
        }

    text = text.strip()

    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.I
    )
    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    # Direct JSON
    try:
        obj = json.loads(text)

        if isinstance(obj, dict):
            label = str(obj.get("label", "")).strip().upper()
            reason = str(obj.get("reason", "")).strip()

            if label in {"SUPPORTED", "UNSUPPORTED"}:
                return {
                    "label": label,
                    "reason": reason
                }
    except Exception:
        pass

    # Extract JSON object from surrounding text
    match = re.search(r"\{.*?\}", text, flags=re.S)

    if match:
        try:
            obj = json.loads(match.group(0))
            label = str(obj.get("label", "")).strip().upper()
            reason = str(obj.get("reason", "")).strip()

            if label in {"SUPPORTED", "UNSUPPORTED"}:
                return {
                    "label": label,
                    "reason": reason
                }
        except Exception:
            pass

    upper = text.upper()

    if "UNSUPPORTED" in upper:
        return {
            "label": "UNSUPPORTED",
            "reason": text
        }

    if "SUPPORTED" in upper:
        return {
            "label": "SUPPORTED",
            "reason": text
        }

    return {
        "label": "UNSUPPORTED",
        "reason": f"Unparseable judge output: {text}"
    }


## 7. Prompt-based Judge function

In [ ]:

@torch.inference_mode()
def judge_claim_prompt(
    claim,
    evidence_text,
    max_new_tokens=160
):
    messages = build_judge_messages(
        claim,
        evidence_text
    )

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=3500
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    generated = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    raw_output = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    parsed = parse_judgment(raw_output)

    return {
        "label": parsed["label"],
        "reason": parsed["reason"],
        "raw_output": raw_output
    }


## 8. Test one claim before the full run

In [ ]:

test_idx = 0

test_row = df.iloc[test_idx]

claim = str(test_row["Atomic_Claim"])
evidence_text = combine_evidence(test_row)

print("CLAIM:")
print(claim)

print("\nEVIDENCE:")
print(evidence_text)

test_result = judge_claim_prompt(
    claim,
    evidence_text
)

print("\nJUDGMENT:")
print(test_result)



## 9. Run the Judge over all atomic claims


In [ ]:

if os.path.exists(CLAIM_OUTPUT_CSV):
    work_df = pd.read_csv(CLAIM_OUTPUT_CSV)
    print("Resuming existing claim-level output.")
else:
    work_df = df.copy()
    work_df["Combined_Evidence"] = ""
    work_df["Prompt_Judge_Label"] = ""
    work_df["Prompt_Judge_Reason"] = ""
    work_df["Prompt_Judge_Raw"] = ""


for idx in tqdm(range(len(work_df))):

    existing = clean_text(
        work_df.at[idx, "Prompt_Judge_Label"]
    )

    if existing in {
        "SUPPORTED",
        "UNSUPPORTED"
    }:
        continue

    claim = str(
        work_df.at[idx, "Atomic_Claim"]
    )

    row = work_df.loc[idx]
    evidence_text = combine_evidence(row)

    try:
        result = judge_claim_prompt(
            claim,
            evidence_text
        )

        work_df.at[idx,"Combined_Evidence"] = evidence_text

        work_df.at[idx,"Prompt_Judge_Label"] = result["label"]

        work_df.at[idx,"Prompt_Judge_Reason"] = result["reason"]

        work_df.at[idx,"Prompt_Judge_Raw"] = result["raw_output"]

    except Exception as e:
        print(
            f"Row {idx} failed:",
            e
        )

        work_df.at[
            idx,
            "Prompt_Judge_Label"
        ] = "UNSUPPORTED"

        work_df.at[
            idx,
            "Prompt_Judge_Reason"
        ] = f"JUDGE ERROR: {e}"

        work_df.at[
            idx,
            "Prompt_Judge_Raw"
        ] = f"ERROR: {e}"

    if (idx + 1) % 10 == 0:
        work_df.to_csv(
            CLAIM_OUTPUT_CSV,
            index=False
        )


work_df.to_csv(
    CLAIM_OUTPUT_CSV,
    index=False
)

print("Saved:", CLAIM_OUTPUT_CSV)


## 10. Inspect claim-level judgments

In [ ]:

display_cols = [
    "Question_ID",
    "Claim_ID",
    "Atomic_Claim",
    "Prompt_Judge_Label",
    "Prompt_Judge_Reason",
]

display(
    work_df[display_cols].head(20)
)



# 11. Simple Aggregator


In [ ]:

def aggregate_answer(group):

    labels = (
        group["Prompt_Judge_Label"]
        .astype(str)
        .str.upper()
        .tolist()
    )

    n_claims = len(labels)
    n_supported = sum(
        x == "SUPPORTED"
        for x in labels
    )
    n_unsupported = sum(
        x == "UNSUPPORTED"
        for x in labels
    )

    unsupported_ratio = (
        n_unsupported / n_claims
        if n_claims > 0
        else np.nan
    )

    # Strict OR-rule baseline
    hallucination = (
        "YES"
        if n_unsupported > 0
        else "NO"
    )

    return pd.Series({
        "Num_Claims": n_claims,
        "Num_Supported": n_supported,
        "Num_Unsupported": n_unsupported,
        "Unsupported_Ratio": unsupported_ratio,
        "Hallucination": hallucination,
    })


## 12. Aggregate to answer level

In [ ]:

answer_results = (
    work_df
    .groupby(
        "Question_ID",
        dropna=False
    )
    .apply(
        aggregate_answer,
        include_groups=False
    )
    .reset_index()
)

# Attach original question and Phi-3 answer.
answer_meta = (
    work_df[
        [
            "Question_ID",
            "Question",
            "Phi3_Answer"
        ]
    ]
    .drop_duplicates(
        subset=["Question_ID"]
    )
)

answer_results = answer_meta.merge(
    answer_results,
    on="Question_ID",
    how="right"
)

answer_results.to_csv(
    ANSWER_OUTPUT_CSV,
    index=False
)

print("Saved:", ANSWER_OUTPUT_CSV)

answer_results.head(20)



## 13. Claim-level statistics

This tells us what percentage of atomic claims the Prompt Judge marks as supported.


In [ ]:

claim_counts = (
    work_df["Prompt_Judge_Label"]
    .value_counts(dropna=False)
)

claim_percent = (
    work_df["Prompt_Judge_Label"]
    .value_counts(
        normalize=True,
        dropna=False
    )
    .mul(100)
    .round(2)
)

claim_stats = pd.DataFrame({
    "Count": claim_counts,
    "Percent": claim_percent
})

print("CLAIM-LEVEL RESULTS")
display(claim_stats)


## 14. Answer-level hallucination statistics

In [ ]:

answer_counts = (
    answer_results["Hallucination"]
    .value_counts(dropna=False)
)

answer_percent = (
    answer_results["Hallucination"]
    .value_counts(
        normalize=True,
        dropna=False
    )
    .mul(100)
    .round(2)
)

answer_stats = pd.DataFrame({
    "Count": answer_counts,
    "Percent": answer_percent
})

print("ANSWER-LEVEL RESULTS")
display(answer_stats)


## 15. Additional summary

In [ ]:

summary = {
    "num_answers": len(answer_results),
    "num_claims": len(work_df),

    "supported_claims": int(
        (work_df["Prompt_Judge_Label"] == "SUPPORTED").sum()
    ),

    "unsupported_claims": int(
        (work_df["Prompt_Judge_Label"] == "UNSUPPORTED").sum()
    ),

    "hallucinated_answers": int(
        (answer_results["Hallucination"] == "YES").sum()
    ),

    "non_hallucinated_answers": int(
        (answer_results["Hallucination"] == "NO").sum()
    ),

    "claim_support_rate": float(
        (work_df["Prompt_Judge_Label"] == "SUPPORTED").mean()
    ),

    "answer_hallucination_rate": float(
        (answer_results["Hallucination"] == "YES").mean()
    ),

    "mean_unsupported_ratio": float(
        answer_results["Unsupported_Ratio"].mean()
    ),
}

for k, v in summary.items():
    print(f"{k}: {v}")



# 16. Optional aggregation ablation — no need to rerun the Judge


In [ ]:

def add_threshold_aggregation(
    answer_df,
    threshold=0.25
):
    result = answer_df.copy()

    result[
        f"Hallucination_Ratio_{threshold}"
    ] = np.where(
        result["Unsupported_Ratio"]
        >= threshold,
        "YES",
        "NO"
    )

    return result

threshold_results = add_threshold_aggregation(
    answer_results,
    threshold=0.25
)

threshold_results[
    [
        "Question_ID",
        "Unsupported_Ratio",
        "Hallucination",
        "Hallucination_Ratio_0.25"
    ]
].head(20)
